In [5]:
df = spark.read.json("transactions_10k.jsonl")

df.show(5)

+------+-----------+--------+-------------------+-------+-------+
|amount|   category|   store|          timestamp|  tx_id|user_id|
+------+-----------+--------+-------------------+-------+-------+
|312.32|elektronika|Warszawa|2026-04-12 08:25:07|TX00001|    u48|
| 79.57|    książki|Warszawa|2026-04-12 08:05:43|TX00002|    u15|
|126.17|     odzież|Warszawa|2026-04-12 09:15:30|TX00003|    u18|
| 34.08|     odzież|Warszawa|2026-04-12 10:05:39|TX00004|    u10|
|428.88|    żywność|  Kraków|2026-04-12 09:04:36|TX00005|    u17|
+------+-----------+--------+-------------------+-------+-------+
only showing top 5 rows



In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Lab2")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

In [6]:
from pyspark.sql.functions import to_timestamp, col

df = df.withColumn(
    "timestamp",
    to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss")
)

df.printSchema()

root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [7]:
from pyspark.sql.functions import avg, window, round as _round

gdansk_hourly = (
    df.filter(col("store") == "Gdańsk")
    .groupBy(window("timestamp", "1 hour"))
    .agg(_round(avg("amount"), 2).alias("srednia_PLN"))
)

gdansk_hourly.show()

+--------------------+-----------+
|              window|srednia_PLN|
+--------------------+-----------+
|{2026-04-12 08:00...|     395.01|
|{2026-04-12 09:00...|     415.91|
|{2026-04-12 10:00...|     412.92|
+--------------------+-----------+



In [ ]:
## zad 1 Znajdź godzinę, w której sklep Gdańsk miał najniższą średnią kwotę transakcji.

In [8]:
from pyspark.sql.functions import avg, window, col, round as _round

zad1 = (
    df.filter(col("store") == "Gdańsk")
    .groupBy(window("timestamp", "1 hour"))
    .agg(_round(avg("amount"), 2).alias("srednia_PLN"))
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "srednia_PLN"
    )
    .orderBy("srednia_PLN")
)

zad1.show(truncate=False)

+-------------------+-------------------+-----------+
|od                 |do                 |srednia_PLN|
+-------------------+-------------------+-----------+
|2026-04-12 08:00:00|2026-04-12 09:00:00|395.01     |
|2026-04-12 10:00:00|2026-04-12 11:00:00|412.92     |
|2026-04-12 09:00:00|2026-04-12 10:00:00|415.91     |
+-------------------+-------------------+-----------+



In [ ]:
## zad 2 Policz ile transakcji per kategoria było w oknie 09:00–09:30.

In [9]:
from pyspark.sql.functions import count

zad2 = (
    df.filter(
        (col("timestamp") >= "2023-01-01 09:00:00") &
        (col("timestamp") <  "2023-01-01 09:30:00")
    )
    .groupBy("category")
    .agg(count("tx_id").alias("liczba_tx"))
    .orderBy("category")
)

zad2.show()

+--------+---------+
|category|liczba_tx|
+--------+---------+
+--------+---------+



In [ ]:
# wniosek: w podanym przedziale nie wystąpiły żadne transakcje

In [ ]:
## zad 3 Okno 15-minutowe — kiedy był szczyt transakcji

In [10]:
from pyspark.sql.functions import desc

zad3 = (
    df.groupBy(window("timestamp", "15 minutes"))
    .agg(count("tx_id").alias("liczba_tx"))
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx"
    )
    .orderBy(desc("liczba_tx"))
)

zad3.show(truncate=False)

+-------------------+-------------------+---------+
|od                 |do                 |liczba_tx|
+-------------------+-------------------+---------+
|2026-04-12 09:15:00|2026-04-12 09:30:00|1234     |
|2026-04-12 09:00:00|2026-04-12 09:15:00|1171     |
|2026-04-12 09:30:00|2026-04-12 09:45:00|1156     |
|2026-04-12 08:45:00|2026-04-12 09:00:00|1139     |
|2026-04-12 09:45:00|2026-04-12 10:00:00|1100     |
|2026-04-12 08:30:00|2026-04-12 08:45:00|899      |
|2026-04-12 10:00:00|2026-04-12 10:15:00|858      |
|2026-04-12 08:15:00|2026-04-12 08:30:00|644      |
|2026-04-12 10:15:00|2026-04-12 10:30:00|582      |
|2026-04-12 08:00:00|2026-04-12 08:15:00|468      |
|2026-04-12 10:30:00|2026-04-12 10:45:00|443      |
|2026-04-12 10:45:00|2026-04-12 11:00:00|306      |
+-------------------+-------------------+---------+

